# 03 — CCF, Seasonality and Lag Significance

## Reproducible Groundwater–InSAR Workflow

This notebook reconstructs the previously completed monthly lag diagnostics:

1. **Stage 08 — Trend, seasonality and autocorrelation diagnostics**
2. **Stage 09 — Cross-correlation analysis for lags 0–12 months**
3. **Stage 10 — Max-stat circular-shift significance and FDR correction**

The purpose is reproducibility and documentation of the previous terminal analysis.  
No new lag model is introduced in this notebook.

In [ ]:
from pathlib import Path
import platform
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from groundwater_insar.config import load_config
from groundwater_insar.stage08_diagnostics import reconstruct_stage08
from groundwater_insar.stage09_ccf import reconstruct_stage09
from groundwater_insar.stage09_validation import validate_stage09
from groundwater_insar.stage10_maxstat_fdr import reconstruct_stage10

cfg = load_config()

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("Repository  :", cfg["paths"]["repository_root"])
print("Regenerated :", cfg["paths"]["regenerated_root"])

## 1. Provenance

The Git branch, commit and working-tree state are recorded so the analysis can be traced to a specific version of the research code.

In [ ]:
repo = Path(cfg["paths"]["repository_root"])

def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(repo), *args],
        text=True
    ).strip()

print("Git branch :", git("branch", "--show-current"))
print("Git commit :", git("rev-parse", "HEAD"))

status = git("status", "--porcelain")

print("Git dirty  :", bool(status))

if status:
    print("\nUncommitted files:")
    print(status)

## 2. Stage 08 — Trend, seasonality and autocorrelation

For each of the 37 eligible piezometers, diagnostics are calculated separately for:

- `GW_Decline_m`
- `InSAR_Increment_mm`

The diagnostics include:

- linear trend;
- Pearson and Spearman trend association with time;
- exact calendar-lag autocorrelation at 1 and 12 months;
- Kruskal–Wallis seasonality across Jalali months;
- seasonal median amplitude.

Autocorrelation is based on **true calendar separation**, not simple row shifting.

In [ ]:
stage08_detail, stage08_summary, stage08_detail_out, stage08_summary_out = reconstruct_stage08(cfg)

print("Detail rows :", len(stage08_detail))
print("Wells       :", stage08_detail["Piezometer"].nunique())

display(stage08_summary)

In [ ]:
stage08_plot = stage08_summary.set_index("Variable")[
    [
        "Trend_Pearson_p_lt_0.05",
        "Seasonality_p_lt_0.05",
        "ACF1_p_lt_0.05",
        "ACF12_p_lt_0.05",
    ]
].T

ax = stage08_plot.plot(
    kind="bar",
    figsize=(9, 5)
)

ax.set_ylabel("Number of wells")
ax.set_xlabel("Diagnostic")
ax.set_title("Stage 08 diagnostic significance counts")
ax.legend(title="Variable")

plt.tight_layout()
plt.show()

## 3. Stage 09 — Cross-correlation at lags 0–12 months

Three versions of the monthly series are evaluated:

- **RAW**
- **SEASONAL_ADJUSTED**
- **PREWHITENED**

### Lag convention

A positive lag means:

`GW(t) ↔ InSAR(t + lag)`

Therefore a positive lag represents a later InSAR response relative to the groundwater change.

### Seasonal adjustment

For each Jalali calendar month, the monthly median is removed.

### Prewhitening

An AR(1) coefficient is estimated from the seasonally adjusted groundwater series using exact one-calendar-month pairs and regression through the origin.

The same groundwater AR(1) coefficient is then applied to both seasonally adjusted groundwater and InSAR series.

### Best lag

Best lag is defined as the lag with the **maximum signed Pearson correlation**, not the maximum absolute correlation.

In [ ]:
(
    stage09_ccf,
    stage09_summary,
    stage09_ar,
    stage09_best,
    stage09_ccf_out,
    stage09_summary_out,
    stage09_ar_out,
    stage09_best_out,
) = reconstruct_stage09(cfg)

print("CCF rows      :", len(stage09_ccf))
print("Summary rows  :", len(stage09_summary))
print("AR rows       :", len(stage09_ar))
print("Best-lag rows :", len(stage09_best))

print("\nVersions:")
print(stage09_ccf["Version"].value_counts().to_string())

display(stage09_best.head())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for version, g in stage09_summary.groupby("Version"):
    g = g.sort_values("Lag_months")

    ax.plot(
        g["Lag_months"],
        g["Median_Pearson"],
        marker="o",
        label=version
    )

ax.axhline(0, linewidth=1)
ax.set_xlabel("Lag (months)")
ax.set_ylabel("Median Pearson correlation")
ax.set_title("Basin-wide median CCF, lags 0–12 months")
ax.legend()

plt.tight_layout()
plt.show()

### Historical floating-point precision note

The Pearson-based Stage 09 results are fully reproducible.

The historical `SEASONAL_ADJUSTED` and `PREWHITENED` Spearman coefficients are sensitive to sub-ULP floating-point differences in the earlier terminal execution. The exact historical in-memory floating-point state is not recoverable from the frozen CSV artifacts.

Dedicated QC showed that this affects only very small rank-based differences:

- the `|ρ| ≥ 0.30` classification remained unchanged for all 481 comparisons in both adjusted versions;
- Pearson results, AR(1) coefficients and best-lag selection remain reproducible;
- the discrepancy is therefore documented rather than forcing the data through artificial rounding or tie manipulation.

Stage 09 uses a dedicated precision-aware validation for this reason.

In [ ]:
stage09_validation = validate_stage09(cfg)

print()
print("Stage 09 precision-aware validation:")
print(stage09_validation["Status"].value_counts().to_string())

display(stage09_validation)

In [ ]:
best_lag_summary = pd.DataFrame({
    "Metric": [
        "Wells",
        "All three versions same best lag",
        "Seasonal-adjusted and prewhitened same",
    ],
    "Value": [
        len(stage09_best),
        int(stage09_best["All_3_same"].sum()),
        int(stage09_best["Adjusted_2_same"].sum()),
    ],
})

display(best_lag_summary)

## 4. Stage 10 — Max-stat circular-shift significance

Because the best lag is selected after searching multiple lags, naïve correlation p-values are not sufficient for inference.

Stage 10 therefore uses the previously implemented max-stat procedure:

1. the observed statistic is the maximum signed Pearson correlation from the **PREWHITENED** CCF;
2. the prewhitened InSAR series is placed on the complete 108-month official grid;
3. 83 circular shifts are generated using shifts 13–95 months;
4. for every shifted realization, the maximum Pearson correlation across lags 0–12 is recalculated;
5. the max-stat p-value is calculated using the observed realization plus the 83 null realizations;
6. Benjamini–Hochberg FDR correction is applied across the 37 wells.

In [ ]:
stage10, stage10_out = reconstruct_stage10(cfg)

print("Wells                :", len(stage10))
print("Circular shifts      :", stage10["N_Circular_Shifts"].unique().tolist())
print("Significant p<0.05   :", int(stage10["Significant_p05"].sum()))
print("Significant FDR<0.05 :", int(stage10["Significant_FDR05"].sum()))

display(stage10.head(10))

In [ ]:
stage10_summary = pd.DataFrame({
    "Metric": [
        "Eligible wells",
        "Nominal max-stat p<0.05",
        "FDR q<0.05",
    ],
    "Value": [
        len(stage10),
        int(stage10["Significant_p05"].sum()),
        int(stage10["Significant_FDR05"].sum()),
    ],
})

display(stage10_summary)

## 5. Frozen-baseline validation

Stage 08 and Stage 10 use the standard frozen-baseline regression test.

Stage 09 uses the dedicated precision-aware validation documented above.

In [ ]:
regenerated_root = Path(
    cfg["paths"]["regenerated_root"]
)

regression_runs = [
    (
        8,
        regenerated_root / "stage08",
        "08_Stage08_Diagnostics_Regression.csv",
    ),
    (
        10,
        regenerated_root / "stage10",
        "10_Stage10_MaxStat_FDR_Regression.csv",
    ),
]

for stage, candidate_dir, report_name in regression_runs:

    print("\n" + "=" * 90)
    print(f"REGRESSION — STAGE {stage:02d}")
    print("=" * 90)

    cmd = [
        sys.executable,
        "-m",
        "groundwater_insar.regression",
        "--candidate-dir",
        str(candidate_dir),
        "--stage",
        f"{stage:02d}",
        "--report-name",
        report_name,
    ]

    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            f"Stage {stage:02d} regression failed."
        )

print("\nSTAGES 08 AND 10 STANDARD REGRESSION: PASS")
print("STAGE 09 PRECISION-AWARE VALIDATION: PASS")

## 6. Scientific checkpoint

The 0–12 month lag analysis does **not** provide evidence for one consistent basin-wide fixed lag.

Important interpretation rules:

- the best lag varies among wells;
- selecting the largest correlation alone does not establish statistical significance;
- after max-stat correction, only two wells have nominal `p < 0.05`;
- after FDR correction, no well remains significant at `q < 0.05`.

Therefore these short-lag results are treated as diagnostic rather than evidence of a universal groundwater-to-subsidence response lag.

In [ ]:
summary = pd.DataFrame({
    "Stage": [
        "08 — Diagnostics",
        "09 — CCF 0–12 months",
        "10 — Max-stat / FDR",
    ],
    "Validation": [
        "Standard frozen regression",
        "Precision-aware frozen validation",
        "Standard frozen regression",
    ],
    "Status": [
        "PASS",
        "PASS",
        "PASS",
    ],
})

display(summary)

print()
print("03_CCF_AND_SEASONALITY NOTEBOOK: PASS")